In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# 1. read files
gt_file = "normal_traj.txt"
est_file = "spoofed_traj.txt"

# 2. Load space-separated .txt data and sort chronologically
# sep=r'\s+' correctly splits the text into separate columns!
df_gt = pd.read_csv(gt_file, sep=r'\s+', header=None).sort_values(by=0).reset_index(drop=True)
df_est = pd.read_csv(est_file, sep=r'\s+', header=None).sort_values(by=0).reset_index(drop=True)

# 3. Synchronize using nearest timestamp matching
df_gt_clean = df_gt[[0, 1, 2]].copy().rename(columns={0: 'timestamp', 1: 'gt_x', 2: 'gt_y'})
df_est_clean = df_est[[0, 1, 2]].copy().rename(columns={0: 'timestamp', 1: 'est_x', 2: 'est_y'})

merged = pd.merge_asof(
    df_gt_clean.sort_values('timestamp'),
    df_est_clean.sort_values('timestamp'),
    on='timestamp',
    direction='nearest'
)

# 4. Calculate error vectors
merged['error_x'] = merged['est_x'] - merged['gt_x']
merged['error_y'] = merged['est_y'] - merged['gt_y']

# 5. Compute 2D Magnitude & 2D Direction Angle (Degrees)
merged['error_magnitude_2d'] = np.sqrt(merged['error_x']**2 + merged['error_y']**2)
merged['error_direction_deg'] = np.degrees(np.arctan2(merged['error_y'], merged['error_x']))

# 6. Print a quick preview of the results
print("\n" + "="*85)
print(f"{'Timestamp':<15} | {'2D Magnitude':<15} | {'Direction Angle':<18} | {'Delta X':<12} | {'Delta Y':<12}")
print("="*85)

for _, row in merged.head(10).iterrows():
    print(f"{row['timestamp']:<15.4f} | {row['error_magnitude_2d']:<15.4f} | {row['error_direction_deg']:+18.2f}° | {row['error_x']:+12.4f} | {row['error_y']:+12.4f}")

print("..." if len(merged) > 10 else "")
print("="*85)

# 7. Save the explicit magnitude and direction output to a CSV file
output_filename = "trajectory_errors_2d_magnitude_direction.csv"
output_df = merged[['timestamp', 'error_magnitude_2d', 'error_direction_deg', 'error_x', 'error_y']]
output_df.to_csv(output_filename, index=False)
print(f"\n[Success] 2D magnitude and direction file saved to '{output_filename}'.")



Timestamp       | 2D Magnitude    | Direction Angle    | Delta X      | Delta Y     
0.1000          | 0.0000          |              +0.00° |      +0.0000 |      +0.0000
0.2000          | 0.6591          |            -179.22° |      -0.6590 |      -0.0090
0.3000          | 0.6821          |            -179.08° |      -0.6820 |      -0.0110
0.5000          | 1.4183          |            -178.87° |      -1.4180 |      -0.0280
0.7000          | 1.4668          |            -178.09° |      -1.4660 |      -0.0490
1.0000          | 0.6778          |            -177.21° |      -0.6770 |      -0.0330
1.2000          | 0.1472          |              -3.12° |      +0.1470 |      -0.0080
1.4000          | 1.5326          |            -176.67° |      -1.5300 |      -0.0890
1.6000          | 1.5461          |            -176.37° |      -1.5430 |      -0.0980
1.8000          | 0.6867          |            -174.90° |      -0.6840 |      -0.0610
...

[Success] 2D magnitude and direction file saved t

In [3]:
import pandas as pd
import numpy as np

# 1. Load the generated 2D error report
try:
    df = pd.read_csv("trajectory_errors_2d_magnitude_direction.csv")
except FileNotFoundError:
    print("Error: Please run the previous cell first to generate the CSV.")
    raise

# 2. Setup window parameters
window_size = 5
num_windows = len(df) // window_size

window_stds_angle = []
window_stds_mag = []

# 3. Process each consecutive block of 5 rows
for i in range(num_windows):
    # Slice the chunk
    window = df.iloc[i * window_size : (i + 1) * window_size].copy()

    # Calculate the initial mean of this specific window
    mean_angle = window['error_direction_deg'].mean()
    mean_mag = window['error_magnitude_2d'].mean()

    # Measure the deviation from the window mean
    # (Using circular distance logic for angles to cleanly handle wrap-arounds near ±180°)
    angle_diff = (window['error_direction_deg'] - mean_angle).abs()
    angle_diff = np.minimum(angle_diff, 360 - angle_diff)

    mag_diff = (window['error_magnitude_2d'] - mean_mag).abs()

    # Filter out rows matching outlier rules (> 45 deg or > 1.0m from the mean)
    filtered_window = window[(angle_diff <= 45) & (mag_diff <= 1.0)]

    # We need at least 2 remaining rows in the window to compute a valid standard deviation
    if len(filtered_window) >= 2:
        window_stds_angle.append(filtered_window['error_direction_deg'].std())
        window_stds_mag.append(filtered_window['error_magnitude_2d'].std())

# 4. Calculate final stats across all processed window standard deviations
final_mean_std_angle = np.mean(window_stds_angle)
final_std_std_angle = np.std(window_stds_angle, ddof=1)  # Standard deviation of the standard deviations

final_mean_std_mag = np.mean(window_stds_mag)
final_std_std_mag = np.std(window_stds_mag, ddof=1)    # Standard deviation of the standard deviations

# 5. Output the metrics
print("=" * 65)
print(f"SUMMARY STATISTICS OF WINDOW STANDARD DEVIATIONS (Window Size = {window_size})")
print(f"Total Windows Evaluated: {num_windows}")
print(f"Valid Windows (post-outlier clean): {len(window_stds_angle)}")
print("=" * 65)
print(f"ANGLE STATISTICS (Degrees, °):")
print(f"  • Mean of Standard Deviations:               {final_mean_std_angle:.4f}°")
print(f"  • Standard Deviation of Standard Deviations: {final_std_std_angle:.4f}°")
print("-" * 65)
print(f"MAGNITUDE STATISTICS (Meters, m):")
print(f"  • Mean of Standard Deviations:               {final_mean_std_mag:.6f} m")
print(f"  • Standard Deviation of Standard Deviations: {final_std_std_mag:.6f} m")
print("=" * 65)

SUMMARY STATISTICS OF WINDOW STANDARD DEVIATIONS (Window Size = 5)
Total Windows Evaluated: 263
Valid Windows (post-outlier clean): 229
ANGLE STATISTICS (Degrees, °):
  • Mean of Standard Deviations:               7.6081°
  • Standard Deviation of Standard Deviations: 8.7452°
-----------------------------------------------------------------
MAGNITUDE STATISTICS (Meters, m):
  • Mean of Standard Deviations:               0.321254 m
  • Standard Deviation of Standard Deviations: 0.211679 m


In [4]:
import pandas as pd
import numpy as np

# 1. Load the generated 2D error report
try:
    df = pd.read_csv("trajectory_errors_2d_magnitude_direction.csv")
except FileNotFoundError:
    print("Error: Please run the previous cell first to generate the CSV.")
    raise

# 2. Setup overlapping window parameters
window_size = 5
# For overlapping windows, the total number of windows is N - window_size + 1
num_windows = len(df) - window_size + 1

window_stds_angle = []
window_stds_mag = []

# 3. Slide the window row-by-row
for i in range(num_windows):
    # Slice the overlapping window of 5 consecutive rows
    window = df.iloc[i : i + window_size].copy()

    # Calculate the initial mean of this window
    mean_angle = window['error_direction_deg'].mean()
    mean_mag = window['error_magnitude_2d'].mean()

    # Measure deviations from the local window mean
    angle_diff = (window['error_direction_deg'] - mean_angle).abs()
    angle_diff = np.minimum(angle_diff, 360 - angle_diff)  # Angular wrap-around correction

    mag_diff = (window['error_magnitude_2d'] - mean_mag).abs()

    # Filter out outliers (> 45 deg or > 1.0m from local mean)
    filtered_window = window[(angle_diff <= 45) & (mag_diff <= 1.0)]

    # We need at least 2 valid points to compute a standard deviation
    if len(filtered_window) >= 2:
        window_stds_angle.append(filtered_window['error_direction_deg'].std())
        window_stds_mag.append(filtered_window['error_magnitude_2d'].std())

# 4. Calculate final stats across all overlapping window standard deviations
final_mean_std_angle = np.mean(window_stds_angle)
final_std_std_angle = np.std(window_stds_angle, ddof=1)  # Std of the obtained standard deviations

final_mean_std_mag = np.mean(window_stds_mag)
final_std_std_mag = np.std(window_stds_mag, ddof=1)    # Std of the obtained standard deviations

# 5. Output the metrics
print("=" * 65)
print(f"SUMMARY STATISTICS FOR OVERLAPPING WINDOWS (Size = {window_size})")
print(f"Total Windows Evaluated: {num_windows}")
print(f"Valid Windows (post-outlier clean): {len(window_stds_angle)}")
print("=" * 65)
print(f"ANGLE STATISTICS (Degrees, °):")
print(f"  • Mean of Standard Deviations:               {final_mean_std_angle:.4f}°")
print(f"  • Standard Deviation of Standard Deviations: {final_std_std_angle:.4f}°")
print("-" * 65)
print(f"MAGNITUDE STATISTICS (Meters, m):")
print(f"  • Mean of Standard Deviations:               {final_mean_std_mag:.6f} m")
print(f"  • Standard Deviation of Standard Deviations: {final_std_std_mag:.6f} m")
print("=" * 65)

SUMMARY STATISTICS FOR OVERLAPPING WINDOWS (Size = 5)
Total Windows Evaluated: 1315
Valid Windows (post-outlier clean): 1127
ANGLE STATISTICS (Degrees, °):
  • Mean of Standard Deviations:               7.8220°
  • Standard Deviation of Standard Deviations: 8.7025°
-----------------------------------------------------------------
MAGNITUDE STATISTICS (Meters, m):
  • Mean of Standard Deviations:               0.319525 m
  • Standard Deviation of Standard Deviations: 0.213915 m


In [5]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load the full CSV file
df = pd.read_csv("trajectory_errors_2d_magnitude_direction.csv")

# 2. Compute Summary Statistics across ALL rows
rmse_mag = np.sqrt((df['error_magnitude_2d'] ** 2).mean())
p95_mag = np.percentile(df['error_magnitude_2d'], 95)

stats_summary = pd.DataFrame({
    "Metric": ["Mean", "Std Dev", "Median", "Min", "Max", "RMSE", "95th Percentile"],
    "2D Magnitude": [
        f"{df['error_magnitude_2d'].mean():.6f}",
        f"{df['error_magnitude_2d'].std():.6f}",
        f"{df['error_magnitude_2d'].median():.6f}",
        f"{df['error_magnitude_2d'].min():.6f}",
        f"{df['error_magnitude_2d'].max():.6f}",
        f"{rmse_mag:.6f}",
        f"{p95_mag:.6f}"
    ],
    "Direction Angle (deg)": [
        f"{df['error_direction_deg'].mean():.2f}°",
        f"{df['error_direction_deg'].std():.2f}°",
        f"{df['error_direction_deg'].median():.2f}°",
        f"{df['error_direction_deg'].min():.2f}°",
        f"{df['error_direction_deg'].max():.2f}°",
        "N/A",
        "N/A"
    ]
})

print("=" * 60)
print("             TRAJECTORY ERROR STATISTICS")
print("=" * 60)
print(stats_summary.to_string(index=False))
print("=" * 60 + "\n")

# 3. Create Plots: Magnitude Histogram & Polar Rose Diagram
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "xy"}, {"type": "polar"}]],
    subplot_titles=("2D Error Magnitude Distribution", "Error Direction Distribution (Polar Rose)")
)

# Plot 1: Histogram of Error Magnitude
fig.add_trace(
    go.Histogram(
        x=df['error_magnitude_2d'],
        nbinsx=30,
        name='Magnitude Count',
        marker_color='royalblue',
        opacity=0.75
    ),
    row=1, col=1
)

# Plot 2: Polar Histogram for Direction Angle
angles_360 = (df['error_direction_deg'] + 360) % 360

fig.add_trace(
    go.Barpolar(
        r=np.histogram(angles_360, bins=36, range=(0, 360))[0],
        theta=np.linspace(0, 360, 36, endpoint=False),
        name='Direction Frequency',
        marker_color='crimson',
        opacity=0.75
    ),
    row=1, col=2
)

# 4. Layout Formatting (FIXED: ticks='outside')
fig.update_layout(
    title="Trajectory Error Distribution Analysis",
    showlegend=False,
    margin=dict(l=40, r=40, b=40, t=60),
    xaxis_title="Magnitude (units)",
    yaxis_title="Frequency",
    polar=dict(
        angularaxis=dict(
            direction="clockwise",
            period=360,
            ticks="outside",  # <-- Fixed here
            tickvals=[0, 45, 90, 135, 180, 225, 270, 315],
            ticktext=['0° (+X)', '45°', '90° (+Y)', '135°', '180° (-X)', '225°', '-90° (-Y)', '315°']
        )
    )
)

fig.show()

             TRAJECTORY ERROR STATISTICS
         Metric 2D Magnitude Direction Angle (deg)
           Mean     4.451816               -55.29°
        Std Dev     3.256010               109.13°
         Median     4.032358              -100.08°
            Min     0.000000              -179.90°
            Max    15.628312               179.92°
           RMSE     5.514728                   N/A
95th Percentile    10.155871                   N/A



In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# 1. Load the CSV file
df = pd.read_csv("trajectory_errors_2d_magnitude_direction.csv")

# 2. Compute Direction Statistics
stats_dir = pd.DataFrame({
    "Metric": ["Mean", "Std Dev", "Median", "Min", "Max"],
    "Direction Angle (deg)": [
        f"{df['error_direction_deg'].mean():.2f}°",
        f"{df['error_direction_deg'].std():.2f}°",
        f"{df['error_direction_deg'].median():.2f}°",
        f"{df['error_direction_deg'].min():.2f}°",
        f"{df['error_direction_deg'].max():.2f}°"
    ]
})

print("=" * 45)
print("     DIRECTIONAL ERROR STATISTICS")
print("=" * 45)
print(stats_dir.to_string(index=False))
print("=" * 45 + "\n")

# 3. Prepare Angles (-180..180 -> 0..360)
angles_360 = (df['error_direction_deg'] + 360) % 360
counts, bin_edges = np.histogram(angles_360, bins=36, range=(0, 360))

# 4. Create Polar Rose Figure
fig = go.Figure()

fig.add_trace(
    go.Barpolar(
        r=counts,
        theta=bin_edges[:-1],
        name='Direction Frequency',
        marker_color='crimson',
        opacity=0.75
    )
)

# 5. Format Layout and Fix Overlap
fig.update_layout(
    title=dict(
        text="Error Direction Distribution (Polar Rose)",
        x=0.5,
        y=0.95,
        xanchor='center',
        yanchor='top',
        font=dict(size=16)
    ),
    showlegend=False,
    margin=dict(l=60, r=60, b=50, t=80),
    polar=dict(
        angularaxis=dict(
            direction="clockwise",
            period=360,
            ticks="outside",
            rotation=90,  # Rotates grid so 0° points right (+X) instead of top
            tickvals=[0, 45, 90, 135, 180, 225, 270, 315],
            ticktext=['0° (+X)', '45°', '90° (+Y)', '135°', '180° (-X)', '225°', '270° (-Y)', '315°']
        ),
        radialaxis=dict(
            angle=45,  # Moves radial scale numbers to 45° to avoid overlapping 0°/90°
            ticks="outside"
        )
    )
)

fig.show()

     DIRECTIONAL ERROR STATISTICS
 Metric Direction Angle (deg)
   Mean               -55.29°
Std Dev               109.13°
 Median              -100.08°
    Min              -179.90°
    Max               179.92°

